In [6]:
import telebot
from telebot import types
import requests
from PIL import Image
import io
import cv2
import numpy as np
#from rembg import remove
from transformers import pipeline
import torch

c:\Users\ThinkBook\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

# Вставьте ваш токен от BotFather
TOKEN = "YOUR TG TOKEN"
bot = telebot.TeleBot(TOKEN)

In [4]:

# Состояния для многошаговых команд (например, для /tryon)
user_states = {}
STATES = {'idle': 0, 'waiting_cloth': 1, 'waiting_person': 2}

# Таблица RAL to RGB (примеры, расширьте по необходимости)
RAL_TO_RGB = {
    'RAL 1000': (190, 183, 146),  # Green beige
    'RAL 1001': (194, 176, 131),  # Beige
    'RAL 1002': (198, 171, 113),  # Sand yellow
    'RAL 1003': (237, 158, 17),   # Signal yellow
    'RAL 1004': (210, 147, 23),   # Golden yellow
    'RAL 1005': (192, 140, 35),   # Honey yellow
    'RAL 1006': (212, 147, 38),   # Maize yellow
    'RAL 1007': (216, 142, 32),   # Daffodil yellow
    'RAL 1011': (160, 124, 81),   # Brown beige
    'RAL 1012': (201, 169, 53),   # Lemon yellow
    'RAL 1013': (221, 210, 182),  # Oyster white
    'RAL 1014': (207, 181, 140),  # Ivory
    'RAL 1015': (217, 198, 163),  # Light ivory
    'RAL 1016': (228, 213, 58),   # Sulfur yellow
    'RAL 1017': (232, 160, 81),   # Saffron yellow
    'RAL 1018': (234, 185, 58),   # Zinc yellow
    'RAL 1019': (153, 127, 102),  # Grey beige
    'RAL 1020': (147, 133, 89),   # Olive yellow
    'RAL 1021': (224, 174, 0),    # Rape yellow
    'RAL 1023': (226, 173, 0),    # Traffic yellow
    'RAL 1024': (166, 133, 78),   # Ochre yellow
    'RAL 1026': (255, 255, 0),    # Luminous yellow
    'RAL 1027': (147, 119, 45),   # Curry
    'RAL 1028': (239, 127, 26),   # Melon yellow
    'RAL 1032': (207, 154, 47),   # Broom yellow
    'RAL 1033': (230, 150, 51),   # Dahlia yellow
    'RAL 1034': (215, 153, 89),   # Pastel yellow
    'RAL 1035': (138, 127, 110),  # Pearl beige
    'RAL 1036': (133, 104, 65),   # Pearl gold
    'RAL 1037': (217, 145, 42),   # Sun yellow
    'RAL 2000': (201, 106, 0),    # Yellow orange
    'RAL 2001': (181, 70, 43),    # Red orange
    'RAL 2002': (176, 61, 46),    # Vermilion
    'RAL 2003': (235, 121, 60),   # Pastel orange
    'RAL 2004': (221, 89, 40),    # Pure orange
    'RAL 2005': (255, 77, 19),    # Luminous orange
    'RAL 2007': (255, 174, 18),   # Luminous bright orange
    'RAL 2008': (222, 105, 51),   # Bright red orange
    'RAL 2009': (221, 93, 45),    # Traffic orange
    'RAL 2010': (198, 95, 61),    # Signal orange
    'RAL 2011': (215, 110, 51),   # Deep orange
    'RAL 2012': (199, 102, 81),   # Salmon orange
    'RAL 2013': (166, 70, 52),    # Pearl orange
    'RAL 3000': (161, 43, 47),    # Flame red
    'RAL 3001': (150, 46, 53),    # Signal red
    'RAL 3002': (142, 50, 56),    # Carmine red
    'RAL 3003': (121, 43, 56),    # Ruby red
    'RAL 3004': (102, 47, 61),    # Purple red
    'RAL 3005': (92, 50, 61),     # Wine red
    'RAL 3007': (61, 43, 49),     # Black red
    'RAL 3009': (102, 63, 63),    # Oxide red
    'RAL 3011': (118, 59, 59),    # Brown red
    'RAL 3012': (184, 133, 117),  # Beige red
    'RAL 3013': (147, 58, 58),    # Tomato red
    'RAL 3014': (194, 118, 129),  # Antique pink
    'RAL 3015': (203, 156, 166),  # Light pink
    'RAL 3016': (153, 74, 71),    # Coral red
    'RAL 3017': (197, 89, 106),   # Rose
    'RAL 3018': (193, 78, 94),    # Strawberry red
    'RAL 3020': (184, 45, 48),    # Traffic red
    'RAL 3022': (199, 102, 92),   # Salmon pink
    'RAL 3024': (255, 42, 50),    # Luminous red
    'RAL 3026': (255, 48, 41),    # Luminous bright red
    'RAL 3027': (156, 58, 81),    # Raspberry red
    'RAL 3028': (198, 55, 61),    # Pure red
    'RAL 3031': (161, 64, 81),    # Orient red
    'RAL 3032': (109, 45, 58),    # Pearl ruby red
    'RAL 3033': (166, 77, 84),    # Pearl pink
    'RAL 4001': (126, 90, 133),   # Red lilac
    'RAL 4002': (133, 68, 92),    # Red violet
    'RAL 4003': (198, 102, 153),  # Heather violet
    'RAL 4004': (99, 50, 78),     # Claret violet
    'RAL 4005': (121, 99, 153),   # Blue lilac
    'RAL 4006': (140, 58, 121),   # Traffic purple
    'RAL 4007': (61, 43, 71),     # Purple violet
    'RAL 4008': (130, 77, 140),   # Signal violet
    'RAL 4009': (150, 127, 140),  # Pastel violet
    'RAL 4010': (179, 71, 133),   # Telemagenta
    'RAL 4011': (117, 99, 135),   # Pearl violet
    'RAL 4012': (99, 99, 122),    # Pearl black berry
    'RAL 5000': (81, 89, 135),    # Violet blue
    'RAL 5001': (35, 71, 102),    # Green blue
    'RAL 5002': (45, 56, 140),    # Ultramarine blue
    'RAL 5003': (40, 56, 89),     # Sapphire blue
    'RAL 5004': (26, 35, 50),     # Black blue
    'RAL 5005': (35, 66, 140),    # Signal blue
    'RAL 5007': (61, 92, 140),    # Brilliant blue
    'RAL 5008': (43, 56, 71),     # Grey blue
    'RAL 5009': (45, 84, 112),    # Azure blue
    'RAL 5010': (35, 71, 117),    # Gentian blue
    'RAL 5011': (30, 45, 66),     # Steel blue
    'RAL 5012': (61, 133, 186),   # Light blue
    'RAL 5013': (35, 45, 76),     # Cobalt blue
    'RAL 5014': (102, 122, 153),  # Pigeon blue
    'RAL 5015': (45, 117, 181),   # Sky blue
    'RAL 5017': (0, 81, 140),     # Traffic blue
    'RAL 5018': (45, 140, 140),   # Turquoise blue
    'RAL 5019': (35, 89, 140),    # Capri blue
    'RAL 5020': (26, 56, 76),     # Ocean blue
    'RAL 5021': (35, 112, 112),   # Water blue
    'RAL 5022': (40, 45, 89),     # Night blue
    'RAL 5023': (81, 112, 153),   # Distant blue
    'RAL 5024': (102, 140, 171),  # Pastel blue
    'RAL 5025': (81, 102, 122),   # Pearl gentian blue
    'RAL 5026': (35, 45, 99),     # Pearl night blue
    'RAL 6000': (61, 135, 102),   # Patina green
    'RAL 6001': (56, 112, 61),    # Emerald green
    'RAL 6002': (56, 102, 50),    # Leaf green
    'RAL 6003': (71, 89, 61),     # Olive green
    'RAL 6004': (35, 81, 76),     # Blue green
    'RAL 6005': (35, 76, 56),     # Moss green
    'RAL 6006': (56, 56, 50),     # Grey olive
    'RAL 6007': (45, 56, 40),     # Bottle green
    'RAL 6008': (50, 50, 40),     # Brown green
    'RAL 6009': (45, 56, 50),     # Fir green
    'RAL 6010': (81, 112, 61),    # Grass green
    'RAL 6011': (112, 135, 92),   # Reseda green
    'RAL 6012': (56, 71, 66),     # Black green
    'RAL 6013': (112, 107, 89),   # Reed green
    'RAL 6014': (71, 66, 56),     # Yellow olive
    'RAL 6015': (56, 61, 50),     # Black olive
    'RAL 6016': (35, 112, 71),    # Turquoise green
    'RAL 6017': (102, 135, 71),   # May green
    'RAL 6018': (102, 153, 61),   # Yellow green
    'RAL 6019': (186, 201, 163),  # Pastel green
    'RAL 6020': (56, 71, 50),     # Chrome green
    'RAL 6021': (135, 153, 112),  # Pale green
    'RAL 6022': (56, 50, 40),     # Brown olive
    'RAL 6024': (45, 135, 92),    # Traffic green
    'RAL 6025': (92, 112, 66),    # Fern green
    'RAL 6026': (35, 92, 76),     # Opal green
    'RAL 6027': (122, 171, 153),  # Light green
    'RAL 6028': (56, 81, 66),     # Pine green
    'RAL 6029': (35, 122, 71),    # Mint green
    'RAL 6032': (81, 140, 92),    # Signal green
    'RAL 6033': (81, 140, 122),   # Mint turquoise
    'RAL 6034': (122, 171, 171),  # Pastel turquoise
    'RAL 6035': (45, 92, 50),     # Pearl green
    'RAL 6036': (35, 99, 71),     # Pearl opal green
    'RAL 6037': (71, 163, 35),    # Pure green
    'RAL 6038': (0, 186, 35),     # Luminous green
    'RAL 7000': (122, 133, 140),  # Squirrel grey
    'RAL 7001': (140, 150, 158),  # Silver grey
    'RAL 7002': (122, 117, 99),   # Olive grey
    'RAL 7003': (117, 117, 107),  # Moss grey
    'RAL 7004': (150, 150, 150),  # Signal grey
    'RAL 7005': (102, 107, 107),  # Mouse grey
    'RAL 7006': (117, 107, 99),   # Beige grey
    'RAL 7008': (107, 99, 76),    # Khaki grey
    'RAL 7009': (92, 97, 89),     # Green grey
    'RAL 7010': (89, 97, 92),     # Tarpaulin grey
    'RAL 7011': (81, 89, 92),     # Iron grey
    'RAL 7012': (81, 89, 92),     # Basalt grey
    'RAL 7013': (76, 71, 61),     # Brown grey
    'RAL 7015': (81, 84, 92),     # Slate grey
    'RAL 7016': (50, 56, 61),     # Anthracite grey
    'RAL 7021': (43, 45, 50),     # Black grey
    'RAL 7022': (66, 66, 61),     # Umbra grey
    'RAL 7023': (107, 112, 107),  # Concrete grey
    'RAL 7024': (56, 61, 66),     # Graphite grey
    'RAL 7026': (56, 66, 71),     # Granite grey
    'RAL 7030': (140, 135, 127),  # Stone grey
    'RAL 7031': (102, 112, 122),  # Blue grey
    'RAL 7032': (186, 181, 166),  # Pebble grey
    'RAL 7033': (127, 133, 122),  # Cement grey
    'RAL 7034': (140, 133, 112),  # Yellow grey
    'RAL 7035': (201, 204, 199),  # Light grey
    'RAL 7036': (153, 150, 150),  # Platinum grey
    'RAL 7037': (122, 122, 122),  # Dusty grey
    'RAL 7038': (179, 176, 171),  # Agate grey
    'RAL 7039': (102, 97, 89),    # Quartz grey
    'RAL 7040': (150, 158, 163),  # Window grey
    'RAL 7042': (150, 156, 158),  # Traffic grey A
    'RAL 7043': (76, 81, 76),     # Traffic grey B
    'RAL 7044': (179, 176, 166),  # Silk grey
    'RAL 7045': (140, 150, 153),  # Telegrey 1
    'RAL 7046': (122, 135, 140),  # Telegrey 2
    'RAL 7047': (204, 204, 201),  # Telegrey 4
    'RAL 7048': (122, 117, 107),  # Pearl mouse grey
    'RAL 8000': (140, 112, 76),   # Green brown
    'RAL 8001': (153, 99, 61),    # Ochre brown
    'RAL 8002': (112, 71, 66),    # Signal brown
    'RAL 8003': (122, 76, 56),    # Clay brown
    'RAL 8004': (140, 81, 66),    # Copper brown
    'RAL 8007': (92, 61, 50),     # Fawn brown
    'RAL 8008': (102, 71, 50),    # Olive brown
    'RAL 8011': (81, 56, 50),     # Nut brown
    'RAL 8012': (92, 50, 50),     # Red brown
    'RAL 8014': (61, 45, 40),     # Sepia brown
    'RAL 8015': (81, 45, 45),     # Chestnut brown
    'RAL 8016': (66, 45, 40),     # Mahogany brown
    'RAL 8017': (56, 45, 45),     # Chocolate brown
    'RAL 8019': (50, 43, 43),     # Grey brown
    'RAL 8022': (35, 30, 30),     # Black brown
    'RAL 8023': (153, 92, 61),    # Orange brown
    'RAL 8024': (112, 81, 66),    # Beige brown
    'RAL 8025': (107, 81, 71),    # Pale brown
    'RAL 8028': (71, 50, 40),     # Terra brown
    'RAL 8029': (107, 61, 56),    # Pearl copper
    'RAL 9001': (237, 232, 219),  # Cream
    'RAL 9002': (219, 215, 204),  # Grey white
    'RAL 9003': (242, 242, 237),  # Signal white
    'RAL 9004': (40, 40, 43),     # Signal black
    'RAL 9005': (10, 10, 13),     # Jet black
    'RAL 9006': (161, 161, 158),  # White aluminium
    'RAL 9007': (140, 140, 135),  # Grey aluminium
    'RAL 9010': (242, 237, 227),  # Pure white
    'RAL 9011': (43, 45, 50),     # Graphite black
    'RAL 9016': (242, 242, 232),  # Traffic white
    'RAL 9017': (40, 40, 40),     # Traffic black
    'RAL 9018': (204, 207, 199),  # Papyrus white
    'RAL 9022': (140, 140, 140),  # Pearl light grey
    'RAL 9023': (122, 122, 122)   # Pearl dark grey
}

In [ ]:
from diffusers import DiffusionPipeline
from diffusers.utils import load_image

tryon_pipe = DiffusionPipeline.from_pretrained("yisol/IDM-VTON")

c:\Users\ThinkBook\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

: 

In [7]:
# Модель для виртуальной примерки (IDM-VTON с Hugging Face)
tryon_pipe = pipeline("image-to-image", model="yisol/IDM-VTON", trust_remote_code=True)

ValueError: Unrecognized model in yisol/IDM-VTON. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: aimv2, aimv2_vision_model, albert, align, altclip, arcee, aria, aria_text, audio-spectrogram-transformer, autoformer, aya_vision, bamba, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, bitnet, blenderbot, blenderbot-small, blip, blip-2, blip_2_qformer, bloom, bridgetower, bros, camembert, canine, chameleon, chinese_clip, chinese_clip_vision_model, clap, clip, clip_text_model, clip_vision_model, clipseg, clvp, code_llama, codegen, cohere, cohere2, colpali, colqwen2, conditional_detr, convbert, convnext, convnextv2, cpmant, csm, ctrl, cvt, d_fine, dab-detr, dac, data2vec-audio, data2vec-text, data2vec-vision, dbrx, deberta, deberta-v2, decision_transformer, deepseek_v2, deepseek_v3, deepseek_vl, deepseek_vl_hybrid, deformable_detr, deit, depth_anything, depth_pro, deta, detr, dia, diffllama, dinat, dinov

In [ ]:


# Команда /start
@bot.message_handler(commands=['start'])
def start(message):
    bot.reply_to(message, "Привет! Я бот для обработки изображений. Доступные команды:\n"
                          "/removebg - Обтравка объекта на белый фон (отправьте фото после команды).\n"
                          "/changecolor - Замена цвета (отправьте фото и текст вроде 'RAL 1001 на RAL 5005').\n"
                          "/tryon - Примерка одежды (отправьте фото одежды, затем фото человека).")

In [ ]:
# Функция скачивания фото из Telegram
def download_photo(file_id):
    file_info = bot.get_file(file_id)
    file = requests.get(f'https://api.telegram.org/file/bot{TOKEN}/{file_info.file_path}')
    return Image.open(io.BytesIO(file.content))

In [ ]:
# 1. Обтравка объекта на белый фон (используя rembg с U2-Net)
@bot.message_handler(commands=['removebg'])
def remove_bg_command(message):
    bot.reply_to(message, "Отправьте фото для обтравки на белый фон.")
    user_states[message.chat.id] = {'state': 'removebg'}

In [ ]:
@bot.message_handler(content_types=['photo'], func=lambda message: user_states.get(message.chat.id, {}).get('state') == 'removebg')
def remove_bg(message):
    photo = message.photo[-1].file_id
    input_image = download_photo(photo)
    
    # Удаление фона с помощью rembg
    output = remove(input_image)
    
    # Установка белого фона
    white_bg = Image.new("RGBA", output.size, (255, 255, 255, 255))
    white_bg.paste(output, (0, 0), output)
    
    # Сохранение и отправка
    img_byte_arr = io.BytesIO()
    white_bg.save(img_byte_arr, format='PNG')
    img_byte_arr.seek(0)
    bot.send_photo(message.chat.id, img_byte_arr)
    user_states[message.chat.id] = {'state': 'idle'}

In [ ]:
# 2. Замена цвета по RAL
@bot.message_handler(commands=['changecolor'])
def change_color_command(message):
    bot.reply_to(message, "Отправьте фото и текст в формате 'с RAL XXXX на RAL YYYY' (например, 'с RAL 1001 на RAL 5005').")
    user_states[message.chat.id] = {'state': 'changecolor'}

In [ ]:

@bot.message_handler(content_types=['photo'], func=lambda message: user_states.get(message.chat.id, {}).get('state') == 'changecolor')
def change_color_photo(message):
    user_states[message.chat.id]['photo'] = message.photo[-1].file_id
    bot.reply_to(message, "Теперь отправьте текст с цветами (например, 'с RAL 1001 на RAL 5005').")

In [ ]:
@bot.message_handler(func=lambda message: user_states.get(message.chat.id, {}).get('state') == 'changecolor' and 'photo' in user_states[message.chat.id])
def change_color_text(message):
    text = message.text.upper()
    if 'С ' not in text or 'НА ' not in text:
        bot.reply_to(message, "Неверный формат. Пример: 'с RAL 1001 на RAL 5005'.")
        return
    
    try:
        from_color = text.split('С ')[1].split(' НА ')[0].strip()
        to_color = text.split('НА ')[1].strip()
        
        if from_color not in RAL_TO_RGB or to_color not in RAL_TO_RGB:
            bot.reply_to(message, "Неизвестный RAL код. Доступные: " + ', '.join(RAL_TO_RGB.keys()))
            return
        
        photo = user_states[message.chat.id]['photo']
        input_image = download_photo(photo)
        input_cv = cv2.cvtColor(np.array(input_image), cv2.COLOR_RGB2BGR)
        
        # Получаем маску объекта с rembg
        mask = remove(input_image, only_mask=True)
        mask = np.array(mask) > 0
        
        # Конвертируем цвета
        from_rgb = np.array(RAL_TO_RGB[from_color], dtype=np.float32) / 255.0
        to_rgb = np.array(RAL_TO_RGB[to_color], dtype=np.float32) / 255.0
        
        # Изменяем цвет в HSV пространстве (простая замена)
        hsv = cv2.cvtColor(input_cv, cv2.COLOR_BGR2HSV)
        # Находим близкие цвета к from_rgb (упрощённо, можно улучшить)
        lower = np.array([0, 0, 0])  # TODO: Улучшить детекцию по from_color
        upper = np.array([179, 255, 255])
        color_mask = cv2.inRange(hsv, lower, upper) & mask.astype(np.uint8) * 255
        
        # Изменяем оттенок
        input_cv[color_mask > 0] = (to_rgb * 255).astype(np.uint8)
        
        # Сохранение и отправка
        result_image = Image.fromarray(cv2.cvtColor(input_cv, cv2.COLOR_BGR2RGB))
        img_byte_arr = io.BytesIO()
        result_image.save(img_byte_arr, format='PNG')
        img_byte_arr.seek(0)
        bot.send_photo(message.chat.id, img_byte_arr)
        
    except Exception as e:
        bot.reply_to(message, f"Ошибка: {str(e)}")
    
    user_states[message.chat.id] = {'state': 'idle'}


In [ ]:
# 3. Примерка одежды (IDM-VTON)
@bot.message_handler(commands=['tryon'])
def tryon_command(message):
    bot.reply_to(message, "Отправьте фото одежды на белом/нейтральном фоне.")
    user_states[message.chat.id] = {'state': 'waiting_cloth'}

@bot.message_handler(content_types=['photo'], func=lambda message: user_states.get(message.chat.id, {}).get('state') == 'waiting_cloth')
def tryon_cloth(message):
    user_states[message.chat.id]['cloth'] = message.photo[-1].file_id
    bot.reply_to(message, "Теперь отправьте фото человека.")
    user_states[message.chat.id]['state'] = 'waiting_person'

@bot.message_handler(content_types=['photo'], func=lambda message: user_states.get(message.chat.id, {}).get('state') == 'waiting_person')
def tryon_person(message):
    cloth_id = user_states[message.chat.id]['cloth']
    person_id = message.photo[-1].file_id
    
    cloth_img = download_photo(cloth_id)
    person_img = download_photo(person_id)
    
    try:
        # Генерация с помощью IDM-VTON (требует torch)
        result = tryon_pipe(person_img, cloth_img, max_length=512)['images'][0]
        
        # Сохранение и отправка
        img_byte_arr = io.BytesIO()
        result.save(img_byte_arr, format='PNG')
        img_byte_arr.seek(0)
        bot.send_photo(message.chat.id, img_byte_arr)
    except Exception as e:
        bot.reply_to(message, f"Ошибка в примерке: {str(e)}. Убедитесь, что фото подходят (одежда на белом фоне).")
    
    user_states[message.chat.id] = {'state': 'idle'}



In [ ]:
# Запуск бота
if __name__ == '__main__':
    bot.infinity_polling()